
# Pairwise sensitivity analysis (real measured second-order damage)

Step 2 (`qnn_and_pruning.ipynb`) only ever measures **one block bypassed at a
time** -- first-order sensitivity. The QUBO's pairwise interaction term
`q_ij = sqrt(C_i*C_j)*(1+0.5*(L_i+L_j))` is a **formula guess** built from that
first-order data, never an actual measurement of two blocks removed together.

This notebook measures the real thing: for **every pair** of the 10
QUBO-selected candidates (`C(10,2) = 45` pairs), bypass **both blocks at once**
and evaluate the real fine-tuned model on real held-out images -- the same
technique as Step 2, just applied to two blocks instead of one.

For each pair we save:
- the **measured** joint damage (loss increase, accuracy drop, F1 drop)
- the **predicted** damage if you naively summed the two blocks' individual
  (first-order) damage -- the QUBO's implicit no-interaction assumption
- the **interaction gap** = measured - predicted (the real second-order effect)
- the **current proxy** `q_ij` formula value, for direct comparison

This is a measurement/reporting step only -- it does **not** modify
`qubo_hamiltonian.py` or re-run QAOA. It produces
`qubo_outputs/pairwise_sensitivity.csv` and a summary JSON so you can see,
pair by pair, how good (or bad) the current formula's guesses actually are.


## Step 1 - Imports and configuration

In [ ]:

from __future__ import annotations

import csv
import gc
import itertools
import json
import math
import time
from pathlib import Path
from typing import Any, Dict, List, Tuple

import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

import timm
from datasets import load_dataset
from huggingface_hub import hf_hub_download

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "qubo_outputs"

CLASS_NAMES = [
    "Calgary", "Charlottetown", "Edmonton", "Halifax", "Hamilton",
    "Kitchener-Waterloo", "Montreal", "Ottawa-Gatineau", "Quebec City",
    "Saskatoon", "St Johns", "Toronto", "Vancouver", "Victoria", "Winnipeg",
]

MODEL_NAME = "convnext"
SPLIT = "test"
MAX_SAMPLES = 600
BATCH_SIZE = 16
NUM_WORKERS = 0

SELECTED_CANDIDATES_CSV = OUTPUT_DIR / "selected_candidates.csv"
COST_LOSS_TABLE_CSV = PROJECT_DIR / "cost_loss_table.csv"
OUTPUT_CSV = OUTPUT_DIR / "pairwise_sensitivity.csv"
OUTPUT_SUMMARY_JSON = OUTPUT_DIR / "pairwise_sensitivity_summary.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project directory:", PROJECT_DIR)
print("Device:", device)


## Step 2 - Basic helpers, preprocessing, dataset, model loading, evaluation

Identical to `classical_pruning.ipynb` / `qnn_and_pruning.ipynb`, duplicated here so this notebook is self-contained.

In [ ]:

def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        if value is None:
            return default
        text = str(value).strip()
        if text == "":
            return default
        return float(text)
    except Exception:
        return default


def safe_int(value: Any, default: int = 0) -> int:
    try:
        if value is None:
            return default
        text = str(value).strip()
        if text == "":
            return default
        return int(float(text))
    except Exception:
        return default


def read_csv(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing CSV file: {path}")
    with path.open("r", newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))


def write_csv(path: Path, rows: List[Dict[str, Any]], fieldnames: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)


def resize_and_pad(img: Image.Image, target_size=(320, 320)) -> Image.Image:
    img = img.copy()
    img.thumbnail(target_size, Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", target_size, (0, 0, 0))
    canvas.paste(img, ((target_size[0] - img.size[0]) // 2,
                       (target_size[1] - img.size[1]) // 2))
    return canvas


CONVNEXT_TRANSFORM = v2.Compose([
    v2.Lambda(lambda img: resize_and_pad(img)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class StreetViewSubset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.data = list(hf_dataset)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        row = self.data[idx]
        img = row["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        img = img.convert("RGB")
        x = self.transform(img)
        y = int(row["label"])
        return x, y


def build_dataloader(model_name, split, max_samples, batch_size, num_workers) -> DataLoader:
    split_expr = f"{split}[:{max_samples}]" if max_samples > 0 else split
    ds = load_dataset("canada-guesser/Canadian-streetview-cities", split=split_expr)
    wrapped = StreetViewSubset(ds, CONVNEXT_TRANSFORM)
    return DataLoader(wrapped, batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=torch.cuda.is_available())


def torch_load_compatible(path: str, device: torch.device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def load_finetuned_model(device: torch.device) -> nn.Module:
    path = hf_hub_download(
        repo_id="canada-guesser/canadian_streetview_cities_models",
        filename="cnn_model/convnext_tiny_set_3_final.bin",
    )
    model = timm.create_model("convnext_tiny", pretrained=False, num_classes=len(CLASS_NAMES))
    checkpoint = torch_load_compatible(path, device)
    state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    total_loss = 0.0
    total = 0
    preds: List[int] = []
    labels: List[int] = []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        if hasattr(logits, "logits"):
            logits = logits.logits
        loss = criterion(logits, y)
        total_loss += float(loss.item())
        total += int(y.numel())
        preds.extend(torch.argmax(logits, dim=1).detach().cpu().tolist())
        labels.extend(y.detach().cpu().tolist())
    return {
        "loss": total_loss / max(total, 1),
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "n_samples": total,
    }


def count_trainable_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def get_module_by_name(model: nn.Module, name: str) -> nn.Module:
    module = model
    for part in name.split("."):
        module = module[int(part)] if part.isdigit() else getattr(module, part)
    return module


def replace_module(model: nn.Module, name: str, new_module: nn.Module) -> None:
    parts = name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = parent[int(part)] if part.isdigit() else getattr(parent, part)
    key = parts[-1]
    if key.isdigit():
        parent[int(key)] = new_module
    else:
        setattr(parent, key, new_module)


## Step 3 - Toggleable bypass wrapper

Same idea as `qnn_and_pruning.ipynb`'s `CandidateWrapper`: wrap a block once, flip a `.bypass` flag to turn pruning on/off, instead of reloading the model or re-wrapping modules per test. This is what makes 45 pairwise evaluations feasible -- one model load, 45 cheap flag toggles, instead of 45 model loads.

In [ ]:

class CandidateWrapper(nn.Module):
    # Wraps a block so it can be bypassed (forward(x) -> x) to simulate pruning.

    def __init__(self, module: nn.Module):
        super().__init__()
        self.module = module
        self.bypass = False

    def forward(self, x, *args, **kwargs):
        return x if self.bypass else self.module(x, *args, **kwargs)


## Step 4 - Load the exact 10 QUBO candidates + their single-block sensitivities

Loaded from `selected_candidates.csv` (not re-discovered from the model) so this experiment's candidate set and naming line up exactly with `interaction_edges.csv` / `hamiltonian_terms.json`. The raw single-block numbers come from `cost_loss_table.csv` -- these are the "predicted = sum of singles" baseline each pair's real measurement gets compared against.

In [ ]:

selected_rows = read_csv(SELECTED_CANDIDATES_CSV)
candidates = []
for row in selected_rows:
    candidates.append({
        "qubit_index": safe_int(row["qubit_index"]),
        "candidate": row["candidate"],
        "params": safe_int(row["params"]),
        "loss_penalty": safe_float(row["loss_penalty"]),
        "compression_value": safe_float(row["compression_value"]),
    })
candidates.sort(key=lambda c: c["qubit_index"])
n = len(candidates)
print(f"Loaded {n} QUBO-selected candidates.")

singles_rows = read_csv(COST_LOSS_TABLE_CSV)
singles_by_name = {r["candidate"]: r for r in singles_rows}

for c in candidates:
    single = singles_by_name.get(c["candidate"])
    if single is None:
        raise ValueError(f"No single-block sensitivity row found for {c['candidate']!r} "
                          f"in {COST_LOSS_TABLE_CSV}")
    c["single_loss_increase_raw"] = safe_float(single["loss_increase_raw"])
    c["single_accuracy_drop_raw"] = safe_float(single["accuracy_drop_raw"])
    c["single_f1_drop_raw"] = safe_float(single["f1_drop_raw"])

for c in candidates:
    print(f"  qubit={c['qubit_index']:>2} {c['candidate']:<20} "
          f"single_loss_inc={c['single_loss_increase_raw']:.4f} "
          f"single_acc_drop={c['single_accuracy_drop_raw']:.4f}")

pairs = list(itertools.combinations(range(n), 2))
print(f"\\nTotal pairs to evaluate: C({n},2) = {len(pairs)}")


## Step 5 - Load model once, wrap all candidates, evaluate the true baseline

In [ ]:

print("Loading validation/test subset...")
loader = build_dataloader(MODEL_NAME, SPLIT, MAX_SAMPLES, BATCH_SIZE, NUM_WORKERS)

print("Loading fine-tuned model...")
model = load_finetuned_model(device)
total_trainable_params = count_trainable_params(model)

wrappers: Dict[int, CandidateWrapper] = {}
for c in candidates:
    original_module = get_module_by_name(model, c["candidate"])
    wrapper = CandidateWrapper(original_module)
    replace_module(model, c["candidate"], wrapper)
    wrappers[c["qubit_index"]] = wrapper

print("Evaluating true baseline (all wrappers bypass=False)...")
baseline_start = time.perf_counter()
baseline_metrics = evaluate(model, loader, device)
baseline_eval_seconds = time.perf_counter() - baseline_start

print(f"Baseline: loss={baseline_metrics['loss']:.6f}, acc={baseline_metrics['accuracy']:.4f}, "
      f"f1={baseline_metrics['macro_f1']:.4f}, eval_time={baseline_eval_seconds:.2f}s")


## Step 6 - Evaluate every pair (bypass both blocks at once)

For each of the 45 pairs: flip both wrappers' `.bypass=True`, evaluate, record, flip back to `False`. The model is never reloaded -- only the two flags change between evaluations.

In [ ]:

result_rows: List[Dict[str, Any]] = []
by_qubit = {c["qubit_index"]: c for c in candidates}

for k, (qi, qj) in enumerate(pairs, start=1):
    ci = by_qubit[qi]
    cj = by_qubit[qj]

    wrappers[qi].bypass = True
    wrappers[qj].bypass = True

    pair_start = time.perf_counter()
    pair_metrics = evaluate(model, loader, device)
    pair_eval_seconds = time.perf_counter() - pair_start

    wrappers[qi].bypass = False
    wrappers[qj].bypass = False

    measured_loss_increase = pair_metrics["loss"] - baseline_metrics["loss"]
    measured_accuracy_drop = baseline_metrics["accuracy"] - pair_metrics["accuracy"]
    measured_f1_drop = baseline_metrics["macro_f1"] - pair_metrics["macro_f1"]

    predicted_loss_increase = ci["single_loss_increase_raw"] + cj["single_loss_increase_raw"]
    predicted_accuracy_drop = ci["single_accuracy_drop_raw"] + cj["single_accuracy_drop_raw"]
    predicted_f1_drop = ci["single_f1_drop_raw"] + cj["single_f1_drop_raw"]

    proxy_q_ij = math.sqrt(ci["compression_value"] * cj["compression_value"]) * (
        1.0 + 0.5 * (ci["loss_penalty"] + cj["loss_penalty"])
    )

    row = {
        "candidate_i": ci["candidate"],
        "candidate_j": cj["candidate"],
        "qubit_i": qi,
        "qubit_j": qj,
        "params_i": ci["params"],
        "params_j": cj["params"],
        "pair_params": ci["params"] + cj["params"],

        "baseline_loss": baseline_metrics["loss"],
        "baseline_accuracy": baseline_metrics["accuracy"],
        "baseline_macro_f1": baseline_metrics["macro_f1"],

        "pair_loss": pair_metrics["loss"],
        "pair_accuracy": pair_metrics["accuracy"],
        "pair_macro_f1": pair_metrics["macro_f1"],

        "measured_loss_increase": measured_loss_increase,
        "measured_accuracy_drop": measured_accuracy_drop,
        "measured_f1_drop": measured_f1_drop,

        "predicted_loss_increase": predicted_loss_increase,
        "predicted_accuracy_drop": predicted_accuracy_drop,
        "predicted_f1_drop": predicted_f1_drop,

        "interaction_loss_gap": measured_loss_increase - predicted_loss_increase,
        "interaction_accuracy_gap": measured_accuracy_drop - predicted_accuracy_drop,
        "interaction_f1_gap": measured_f1_drop - predicted_f1_drop,

        "proxy_q_ij": proxy_q_ij,
        "eval_seconds": pair_eval_seconds,
    }
    result_rows.append(row)

    print(
        f"[{k:>2}/{len(pairs)}] {ci['candidate']:<20} + {cj['candidate']:<20} "
        f"measured_acc_drop={measured_accuracy_drop:+.4f} "
        f"predicted={predicted_accuracy_drop:+.4f} "
        f"gap={row['interaction_accuracy_gap']:+.4f} "
        f"proxy_q_ij={proxy_q_ij:.4f} "
        f"({pair_eval_seconds:.1f}s)"
    )

    del pair_metrics
    gc.collect()

print(f"\\nCompleted {len(result_rows)} pairwise evaluations.")


## Step 7 - Save results

In [ ]:

fieldnames = list(result_rows[0].keys())
write_csv(OUTPUT_CSV, result_rows, fieldnames)

acc_gaps = [r["interaction_accuracy_gap"] for r in result_rows]
loss_gaps = [r["interaction_loss_gap"] for r in result_rows]
underestimated = sum(1 for g in acc_gaps if g > 0)   # actual damage worse than predicted
overestimated = sum(1 for g in acc_gaps if g < 0)     # actual damage better than predicted

proxy_vals = [r["proxy_q_ij"] for r in result_rows]
mean_proxy = sum(proxy_vals) / len(proxy_vals)
mean_gap = sum(acc_gaps) / len(acc_gaps)
# simple Pearson correlation between proxy_q_ij and the real interaction gap
n_pairs = len(result_rows)
mean_p, mean_g = mean_proxy, mean_gap
cov = sum((p - mean_p) * (g - mean_g) for p, g in zip(proxy_vals, acc_gaps)) / n_pairs
std_p = (sum((p - mean_p) ** 2 for p in proxy_vals) / n_pairs) ** 0.5
std_g = (sum((g - mean_g) ** 2 for g in acc_gaps) / n_pairs) ** 0.5
correlation = cov / (std_p * std_g) if std_p > 0 and std_g > 0 else None

top_gaps = sorted(result_rows, key=lambda r: r["interaction_accuracy_gap"], reverse=True)[:5]

summary = {
    "n_pairs": n_pairs,
    "max_samples": MAX_SAMPLES,
    "baseline": baseline_metrics,
    "baseline_eval_seconds": baseline_eval_seconds,
    "total_trainable_params": total_trainable_params,
    "mean_interaction_accuracy_gap": mean_gap,
    "mean_interaction_loss_gap": sum(loss_gaps) / len(loss_gaps),
    "max_interaction_accuracy_gap": max(acc_gaps),
    "min_interaction_accuracy_gap": min(acc_gaps),
    "pairs_where_proxy_underestimated_damage": underestimated,
    "pairs_where_proxy_overestimated_damage": overestimated,
    "correlation_proxy_q_ij_vs_measured_gap": correlation,
    "top_5_largest_measured_interaction_gaps": [
        {
            "candidate_i": r["candidate_i"], "candidate_j": r["candidate_j"],
            "interaction_accuracy_gap": r["interaction_accuracy_gap"],
            "measured_accuracy_drop": r["measured_accuracy_drop"],
            "predicted_accuracy_drop": r["predicted_accuracy_drop"],
            "proxy_q_ij": r["proxy_q_ij"],
        }
        for r in top_gaps
    ],
    "note": (
        "measured_* comes from actually bypassing both blocks together on the real "
        "model. predicted_* is the naive sum of each block's individually-measured "
        "(first-order) damage from cost_loss_table.csv -- the QUBO's implicit "
        "no-interaction assumption. interaction_*_gap = measured - predicted is the "
        "real, measured second-order effect. proxy_q_ij is the CURRENT formula "
        "qubo_hamiltonian.py uses for the pairwise penalty -- compare it against "
        "interaction_accuracy_gap to see how good the formula's guess actually is. "
        "This notebook only measures and reports; it does not modify qubo_hamiltonian.py."
    ),
}
write_json(OUTPUT_SUMMARY_JSON, summary)

print(f"Saved {n_pairs} pairwise rows to: {OUTPUT_CSV}")
print(f"Saved summary to: {OUTPUT_SUMMARY_JSON}")
print(f"\\nMean interaction accuracy gap: {mean_gap:+.4f}")
print(f"Pairs where formula UNDERESTIMATED real damage: {underestimated}/{n_pairs}")
print(f"Pairs where formula OVERESTIMATED real damage: {overestimated}/{n_pairs}")
print(f"Correlation(proxy_q_ij, measured_gap): {correlation}")
print("\\nTop 5 largest measured interaction gaps (biggest hidden risk the proxy missed):")
for r in top_gaps:
    print(f"  {r['candidate_i']} + {r['candidate_j']}: "
          f"gap={r['interaction_accuracy_gap']:+.4f}  "
          f"(measured={r['measured_accuracy_drop']:+.4f}, "
          f"predicted={r['predicted_accuracy_drop']:+.4f}, "
          f"proxy_q_ij={r['proxy_q_ij']:.4f})")
